# v1.2 Root-Anchored24 · 2×2 pilot
학교 계정으로 로그인하고 T4에서 순서대로 실행한다. 모델·이전 RR ZIP·audit·원본 입력 검증 후 세 gradient 진단 → 4×24 updates → 기존 A/B 대조 → 4개 clip 평가 순서다. A100에서 실행 시 이전 T4와의 엄격 재현 검사를 동일하게 적용한다.

In [ ]:
import os,sys,json,hashlib,subprocess,tempfile,zipfile,shutil,time
from pathlib import Path
from datetime import datetime,timezone
from functools import lru_cache
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import torch
auth.authenticate_user()
api=build('drive','v3',credentials=google.auth.default()[0])
school='darkk0729@ajou.ac.kr'
def school_account():
    email=api.about().get(fields='user(emailAddress)').execute()['user']['emailAddress']
    if email.lower()!=school:
        raise PermissionError('학교 계정으로 인증해야 합니다')
school_account()
if not torch.cuda.is_available() or not any(s in torch.cuda.get_device_name() for s in ('T4','A100')):
    raise RuntimeError('T4 or A100 required')
work=Path(tempfile.mkdtemp(dir='/content')); repo=work/'fork'; package=work/'package'
stage=work/'evidence';stage.mkdir()
revision='ed1c1e1ebda25d7f1e5cb495c5e34f06829ac2d2'
def digest(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f,'sha256').hexdigest()
def download(fid,path,pin):
    media=api.files().get_media(fileId=fid,supportsAllDrives=True)
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('xb') as stream:
        reader=MediaIoBaseDownload(stream,media)
        done=False
        while not done: _,done=reader.next_chunk(num_retries=3)
    if digest(path)!=pin:raise ValueError('SHA mismatch: '+path.name)
import traceback
from contextlib import contextmanager
def archive(status):
    school_account()
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    path=Path(shutil.make_archive(str(work/('v12_anchored_'+status+'_'+stamp)),'zip',stage))
    saved=api.files().create(body={'name':path.name,'parents':['root']},media_body=MediaFileUpload(str(path),mimetype='application/zip',resumable=True),fields='id,webViewLink,size,sha256Checksum').execute(num_retries=3)
    pin=digest(path)
    if int(saved['size'])!=path.stat().st_size or saved['sha256Checksum']!=pin:
        raise IOError('Drive archive metadata mismatch')
    readback=work/('readback_'+stamp+'.zip')
    download(saved['id'],readback,pin)
    print(saved['webViewLink'],pin,flush=True)

@contextmanager
def preserve(stage_name):
    try:
        yield
    except BaseException:
        (stage/'failure.txt').write_text(stage_name+'\n'+traceback.format_exc())
        archive('failed')
        raise

with preserve('model and CPU preflight'):
    subprocess.run(['git','clone','-q','--branch','experiment/v12-root-anchored-diffusion-20261122','https://github.com/darkkyoung/REWIND-like-prototype.git',str(repo)],check=True)
    subprocess.run(['git','checkout','--detach',revision],cwd=repo,check=True)
    download('1eXSpkVdoLFybz6QBvX-uyrOFfZURfxdt',work/'model.zip','fbf1adda7fb63007d799545278e513be7f0a32ae09a9453bad70f675312274a0')
    with zipfile.ZipFile(work/'model.zip') as z:
        if z.testzip() is not None:raise ValueError('model CRC')
        z.extractall(package)
    source=package/'source'
    env=dict(os.environ,PYTHONPATH=str(source)+os.pathsep+str(repo),CUBLAS_WORKSPACE_CONFIG=':4096:8')
    def run(*args,log):
        with (stage/log).open('w') as f:
            subprocess.run([sys.executable,*map(str,args)],cwd=source,env=env,stdout=f,stderr=subprocess.STDOUT,check=True)
    run('-m','unittest','-v','experiments.v12_anchored.test_anchor','experiments.v12_rr.test_rr','experiments.v12_m2.test_integration',log='tests.log')
    run('-m','experiments.v12_m2.preflight',package,log='strict_cpu.log')
    print((stage/'strict_cpu.log').read_text())

In [ ]:
with preserve('audit and selected RGB/pose'):
    previous=work/'previous_rr.zip'
    download('16zTnwKQ6Ybo6KSCrotuCdd2BpUPFNxu2',previous,'c1985fe1624489a58df446c77fa42a469a14a3eb92bf3c19b9d12cb67a848eed')
    audit=work/'reports.zip'
    download('1bvmfWhFVMGOaFQDidxp_0KQ12seBfFEs',audit,'30c46b8d3259d9a48eae7f4b94465a5563e3ad6bf4fb74b6b88ef546da8e4c2f')
    sys.path[:0]=[str(source),str(repo)]
    from experiments.v12_anchored.data import selection,HERE
    picked,manifest=selection(audit)
    assert picked==json.loads((HERE/'selection.json').read_text())
    roots={'rgb':'1GdhUa-s7Ov_fFGGOSrBona6X5rQtiQvb','pose':'1G8X753DlLHBI1vBYLYifZcUXPKuKqWkL'}
    @lru_cache(None)
    def listing(parent):
        names={};token=None
        while True:
            page=api.files().list(q=f"'{parent}' in parents and trashed=false",fields='nextPageToken,files(id,name)',pageToken=token,pageSize=1000,includeItemsFromAllDrives=True,supportsAllDrives=True).execute(num_retries=3)
            for item in page['files']:names.setdefault(item['name'],[]).append(item['id'])
            token=page.get('nextPageToken')
            if not token:return names
    rows={(r['sequence_id'],r['frame_index']):r for r in manifest['records']}
    for split in ('train','validation'):
        for seq,start in picked[split]:
            for frame in range(start,start+50):
                row=rows[seq,frame]
                assert row['split']==split
                for role,pin in row['files'].items():
                    kind='pose' if role=='pose' else 'rgb'
                    fid=roots[kind]
                    for part in Path(pin['path']).parts:
                        ids=listing(fid)[part]
                        if len(ids)!=1:raise ValueError('ambiguous path')
                        fid=ids[0]
                    download(fid,work/kind/pin['path'],pin['sha256'])
            print(split,seq,start,flush=True)
    (stage/'inputs.json').write_text(json.dumps(dict(selection=picked,code=revision,package_sha256=digest(work/'model.zip'),audit_sha256=digest(audit),device=torch.cuda.get_device_name(),torch=str(torch.__version__)),indent=2))

In [ ]:
with preserve('diagnose, train and evaluate'):
    run('-m','experiments.v12_anchored.run','--package',package,'--previous',previous,'--audit',audit,'--rgb',work/'rgb','--pose',work/'pose','--out',stage/'results',log='pilot.log')
archive('completed')
print((stage/'results/evaluation.json').read_text())